# pandas: подробный разбор Series, DataFrame, индексации, новых столбцов и объединения таблиц

В исходном файле были показаны:
- создание `Series` и явного индекса;
- `DatetimeIndex` и выравнивание Series по индексам;
- создание `DataFrame` из Series;
- выбор столбцов;
- выбор строк через `.iloc` и `.loc`;
- фильтрация по булевому условию;
- чтение `GOOG.csv`, преобразование `Date` в даты и назначение `Date` индексом;
- создание нового столбца `Difference`.

В этом notebook эти идеи систематизированы в четыре большие темы:
1. Инициализация `Series` и `DataFrame`.
2. Отбор данных и работа с индексами.
3. Создание и преобразование новых столбцов.
4. Объединение нескольких `DataFrame`: `concat`, `merge`, `join`.


In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 15)

print("pandas:", pd.__version__)


## 1. Инициализация `Series`

`Series` — одномерная структура данных с индексом. В исходном notebook сначала использовался обычный `RangeIndex`, а затем индекс задавался явно.

Важно различать **значение** и **метку индекса**. Индекс участвует не только в выборе элементов, но и в выравнивании данных при арифметических операциях.

Официально `Series` поддерживает создание из array-like объектов, iterable, `dict` или scalar; если индекс не задан, используется `RangeIndex`.


In [ ]:
# 1.1 Series из списка — индекс создаётся автоматически
s_default = pd.Series([10, 20, 30, 40])
s_default


In [ ]:
# 1.2 Series с явно заданным индексом
s_named = pd.Series(
    [10, 20, 30, 40],
    index=["a", "b", "c", "d"],
    name="score"
)
s_named


In [ ]:
# 1.3 Series из словаря
s_dict = pd.Series({"apple": 120, "banana": 80, "orange": 100})
s_dict


In [ ]:
# 1.4 Series с датами в качестве индекса
dates = pd.date_range("2026-01-01", periods=5, freq="D")

temps = pd.Series(
    [80, 82, 85, 90, 83],
    index=dates,
    name="temperature"
)

temps


### Индекс — это часть структуры данных

`Series.index` возвращает объект `Index`. Индекс может быть числовым, строковым, датой и т.д.

In [ ]:
print("index:", temps.index)
print("name:", temps.name)
print("dtype:", temps.dtype)


In [ ]:
# Демонстрация выравнивания по индексам
a = pd.Series([100, 200, 300], index=["x", "y", "z"])
b = pd.Series([1, 2, 3], index=["z", "x", "y"])

print("a:")
display(a)

print("b:")
display(b)

print("a - b:")
display(a - b)


## 2. Инициализация `DataFrame`

`DataFrame` — двумерная табличная структура. У него есть:
- индекс строк (`df.index`);
- имена столбцов (`df.columns`);
- значения (`df.values`);
- типы данных столбцов (`df.dtypes`).

Исходный notebook создавал `DataFrame` из нескольких `Series`, используя их индексы для выравнивания. Это очень важная модель мышления pandas.


In [ ]:
# 2.1 DataFrame из словаря списков
df_basic = pd.DataFrame({
    "city": ["Tokyo", "Osaka", "Kyoto"],
    "temperature": [24, 27, 22],
    "humidity": [60, 55, 70],
})
df_basic


In [ ]:
# 2.2 DataFrame из нескольких Series.
# pandas выровняет Series по индексам.
city_a = pd.Series([80, 82, 85], index=["2026-01-01", "2026-01-02", "2026-01-03"])
city_b = pd.Series([70, 75, 69], index=["2026-01-01", "2026-01-02", "2026-01-03"])

weather = pd.DataFrame({
    "Missoula": city_a,
    "Philadelphia": city_b,
})

weather


In [ ]:
# 2.3 Явный индекс DataFrame
df_indexed = pd.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie"],
        "age": [25, 30, 35],
    },
    index=[101, 102, 103]
)

df_indexed


## 3. Загрузка реального `GOOG.csv`

В исходном notebook CSV читался через `pd.read_csv`, затем столбец `Date` преобразовывался в даты, а после этого становился индексом.

Здесь используется загруженный вместе с заданием `GOOG.csv`.


In [ ]:
goog = pd.read_csv(os.path.join('data', "GOOG.csv"))

print("Размер:", goog.shape)
print("Столбцы:", goog.columns.tolist())
display(goog.head())

goog.info()

In [ ]:
# Преобразуем Date сразу при чтении
goog = pd.read_csv(os.path.join('data', "GOOG.csv"), parse_dates=["Date"])

display(goog.head())
print(goog.dtypes)


In [ ]:
# Делаем Date индексом — тот же приём, который использовался
# в исходном notebook.
goog = pd.read_csv(
    os.path.join('data', "GOOG.csv"),
    parse_dates=["Date"],
    index_col="Date"
)

display(goog.head())
print("Тип индекса:", type(goog.index).__name__)
print("Индекс:", goog.index[:5])


# 4. Отбор данных

В pandas нужно чётко различать несколько способов доступа:

| Задача | Пример |
|---|---|
| Столбец | `df["Close"]` |
| Несколько столбцов | `df[["Open", "Close"]]` |
| Строка по метке | `df.loc[label]` |
| Строка по позиции | `df.iloc[position]` |
| Строки по условию | `df[df["Close"] > 800]` |
| Строки + столбцы по меткам | `df.loc[rows, cols]` |
| Строки + столбцы по позициям | `df.iloc[rows, cols]` |

`.loc` — преимущественно label-based, а `.iloc` — integer-position based.


In [ ]:
# 4.1 Один столбец -> Series
close = goog["Close"]
display(close.head())
print(type(close))


In [ ]:
# 4.2 Несколько столбцов -> DataFrame
prices = goog[["Open", "High", "Low", "Close"]]
display(prices.head())
print(type(prices))


In [ ]:
# 4.3 Доступ через точку.
display(goog.Close.head())

## 5. Отбор по индексам: `.loc` и `.iloc`

### `.loc`
Используется, когда мы говорим: **«дай строку/столбец с такой меткой»**.

### `.iloc`
Используется, когда мы говорим: **«дай строку/столбец с такой позицией»**.

В исходном notebook оба варианта демонстрировались на `DatetimeIndex`: `.loc['2016-04-05']` выбирал строку по дате, а `.iloc[1]` — вторую строку по позиции.


In [ ]:
# .loc: строка по метке индекса
first_date = goog.index[0]
display(goog.loc[first_date])


In [ ]:
# .iloc: строка по физической позиции
display(goog.iloc[0])


In [ ]:
# .loc: диапазон дат.
# Для label-based slice конечная дата включается, если она присутствует в индексе.
start = goog.index[0]
end = goog.index[4]

display(goog.loc[start:end, ["Open", "Close"]])


In [ ]:
# .iloc: позиции строк 0, 2, 4 и столбцов 0, 3
display(goog.iloc[[0, 2, 4], [0, 3]])


In [ ]:
# .loc может одновременно выбирать строки и столбцы
display(
    goog.loc[
        goog["Close"] > goog["Open"],
        ["Open", "Close", "Volume"]
    ].head(10)
)


## 6. Булева фильтрация

Булева маска — это `Series` из `True`/`False`. Она позволяет отобрать строки, для которых условие истинно.

Исходный notebook использовал конструкцию `temps_df[temps_df.Missoula > 82]`. Здесь применяем тот же принцип к реальным биржевым данным.


In [ ]:
mask = goog["Close"] > 800

display(mask.head(10))


In [ ]:
# Оставляем только строки, где Close > 800
high_close = goog[goog["Close"] > 800]

display(high_close.head())


In [ ]:
# Несколько условий: используем &, | и скобки
filtered = goog[
    (goog["Close"] > 800) &
    (goog["Volume"] > 1_000_000)
]

display(filtered.head(10))


### Частая ошибка

Нельзя писать:

```python
goog["Close"] > 800 and goog["Volume"] > 1_000_000
```

Для поэлементных условий pandas используются `&` и `|`, а каждое условие желательно заключать в скобки.

# 7. Создание новых столбцов

Новый столбец обычно создаётся присваиванием:

```python
df["new_column"] = ...
```
Если справа находится `Series`, pandas выравнивает значения по индексу.

Есть несколько полезных способов создания столбцов:
1. арифметика существующих столбцов (векторизация);
2. условие;
3. `.assign()`;
4. `map()` / `replace()` для преобразования значений;
5. `apply()` — когда действительно нужна функция по строкам/элементам.
6. `iterrows()`, с помощью циклов


Векторизация — это подход к обработке данных, при котором операции выполняются одновременно над целыми массивами (векторами) данных, а не поэлементно в циклах. В Pandas это означает, что вы применяете операции ко всему столбцу (Series) или DataFrame целиком, используя внутренние оптимизированные C-реализации.


In [ ]:
# 7.1 Простой вычисляемый столбец
goog["Daily_Range"] = goog["High"] - goog["Low"]

display(goog[["High", "Low", "Daily_Range"]].head())


In [ ]:
# 7.2 Доходность внутри дня в процентах
goog["Intraday_Return_%"] = (goog["Close"] - goog["Open"]) / goog["Open"] * 100

display(goog[["Open", "Close", "Intraday_Return_%"]].head())


In [ ]:
# 7.3 Условный столбец
goog["Up_Day"] = goog["Close"] > goog["Open"]

display(goog[["Open", "Close", "Up_Day"]].head())


In [ ]:
# 7.4 Условный текстовый столбец через np.where
goog["Direction"] = np.where(
    goog["Close"] >= goog["Open"],
    "UP",
    "DOWN"
)

display(goog[["Open", "Close", "Direction"]].head())


In [ ]:
# 7.5 Несколько новых столбцов через assign()
# assign удобен для цепочек преобразований.
goog2 = goog.assign(
    Range_Pct=lambda x: (x["High"] - x["Low"]) / x["Open"] * 100,
    Close_to_High=lambda x: x["Close"] / x["High"],
)

display(
    goog2[
        ["Open", "High", "Low", "Close", "Range_Pct", "Close_to_High"]
    ].head()
)


## 8. Изменение существующих столбцов и `dtype`

При создании нового столбца важно контролировать тип данных. Проверить типы можно через `df.dtypes`.

В исходном notebook отдельно демонстрировалось, что без `parse_dates` значение `Date` было строкой, а с `parse_dates=["Date"]` оно стало `Timestamp`.


In [ ]:
print(goog.dtypes)

In [ ]:
# Пример преобразования типа
goog["Volume"] = goog["Volume"].astype("int64")
print(goog["Volume"].dtype)

# 9. Объединение DataFrame

pandas предоставляет несколько механизмов объединения:

- `pd.concat()` — поставить таблицы друг под другом или рядом;
- `pd.merge()` — SQL-подобное объединение по ключам;
- `DataFrame.join()` — удобное объединение по индексам;
- также существуют специализированные `merge_ordered()` и `merge_asof()`.

Официальная документация разделяет эти операции именно по такой логике.


## 10. `concat`: объединение «одинаковых» таблиц

Если есть несколько DataFrame с одинаковой структурой и нужно собрать их в одну таблицу, обычно используется `pd.concat()`.

По умолчанию `axis=0`, то есть строки добавляются друг под другом.


In [ ]:
sales_jan = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-02"],
    "product": ["A", "B"],
    "sales": [100, 150],
})

sales_feb = pd.DataFrame({
    "date": ["2026-02-01", "2026-02-02"],
    "product": ["A", "C"],
    "sales": [120, 180],
})

display(sales_jan)
display(sales_feb)


In [ ]:
sales_all = pd.concat([sales_jan, sales_feb], ignore_index=True)
display(sales_all)


### `axis=1`: объединение по горизонтали

`concat(..., axis=1)` добавляет столбцы. Здесь важен индекс: pandas выравнивает объекты по индексам.


In [ ]:
left = pd.DataFrame(
    {"temperature": [20, 21, 22]},
    index=["Mon", "Tue", "Wed"]
)

right = pd.DataFrame(
    {"rain_mm": [0, 5, 2]},
    index=["Mon", "Tue", "Wed"]
)

display(pd.concat([left, right], axis=1))


## 11. `merge`: объединение по ключу

`merge` похож на SQL JOIN: выбирается один или несколько ключей, по которым строки сопоставляются.

Основные варианты `how`:
- `inner` — только совпавшие ключи;
- `left` — все ключи из левой таблицы;
- `right` — все ключи из правой;
- `outer` — объединение всех ключей;
- `cross` — декартово произведение.


In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "Diana"],
})

orders = pd.DataFrame({
    "customer_id": [1, 1, 2, 5],
    "amount": [100, 250, 80, 300],
})

display(customers)
display(orders)


In [ ]:
# INNER JOIN — только совпавшие customer_id
inner = customers.merge(
    orders,
    on="customer_id",
    how="inner"
)

display(inner)


In [ ]:
# LEFT JOIN — все клиенты сохраняются,
# даже если у них нет заказа.
left_join = customers.merge(
    orders,
    on="customer_id",
    how="left"
)

display(left_join)


In [ ]:
# OUTER JOIN — сохраняются ключи обеих таблиц
outer_join = customers.merge(
    orders,
    on="customer_id",
    how="outer",
    indicator=True
)

display(outer_join)


### `validate`: проверка ожидаемой кардинальности

Очень полезная особенность `merge` — `validate`. Она позволяет явно указать, какие отношения между ключами ожидаются.

Например, `one_to_many` означает: один ключ в левой таблице может соответствовать нескольким строкам справа.

Это помогает обнаруживать неожиданные дубли при объединении.


In [ ]:
# Здесь каждый customer_id в customers уникален,
# а в orders один клиент может иметь несколько заказов.
validated = customers.merge(
    orders,
    on="customer_id",
    how="left",
    validate="one_to_many"
)

display(validated)


## 12. `merge` по разным именам ключей

Ключи не обязаны называться одинаково. Используются `left_on` и `right_on`.


In [ ]:
employees = pd.DataFrame({
    "employee_id": [10, 20, 30],
    "name": ["Alice", "Bob", "Charlie"],
})

salaries = pd.DataFrame({
    "worker_id": [10, 20, 30],
    "salary": [100_000, 120_000, 110_000],
})

display(
    employees.merge(
        salaries,
        left_on="employee_id",
        right_on="worker_id",
        how="left"
    )
)


## 13. `join`: объединение по индексу

`DataFrame.join()` особенно удобен, когда ключ уже находится в индексе.

Это отличается от `merge`, где чаще явно указывают столбцы-ключи. Документация pandas описывает `DataFrame.join()` как способ объединения DataFrame по индексам/с использованием индекса.


In [ ]:
profile = pd.DataFrame(
    {"name": ["Alice", "Bob", "Charlie"]},
    index=[101, 102, 103]
)

salary = pd.DataFrame(
    {"salary": [100_000, 120_000, 110_000]},
    index=[101, 102, 103]
)

display(profile.join(salary))


# 15. Практический микро-пример на GOOG

Соединим несколько изученных операций в один pipeline:
1. читаем CSV;
2. преобразуем дату;
3. делаем дату индексом;
4. создаём новые признаки;
5. фильтруем данные;
6. выбираем нужные столбцы.


In [ ]:
goog_demo = pd.read_csv(
    os.path.join('data', "GOOG.csv"),
    parse_dates=["Date"],
    index_col="Date"
)

result = (
    goog_demo.assign(
        Daily_Range=lambda x: x["High"] - x["Low"],
        Intraday_Return_Pct=lambda x: (
            (x["Close"] - x["Open"]) / x["Open"] * 100
        ),
        Up_Day=lambda x: x["Close"] > x["Open"],
    ).loc[
        lambda x: x["Close"] > x["Open"],
        ["Open", "High", "Low", "Close", "Volume",
         "Daily_Range", "Intraday_Return_Pct", "Up_Day"]
    ]
)

display(result.head(10))


# 16. Краткая шпаргалка

```python
# Series
s = pd.Series([1, 2, 3])
s = pd.Series([1, 2, 3], index=["a", "b", "c"])
s = pd.Series({"a": 1, "b": 2})

# DataFrame
df = pd.DataFrame({"A": [1, 2], "B": [3, 4]})

# Индекс
df.index
df.columns
df.set_index("A")
df.reset_index()

# Выбор
df["A"]
df[["A", "B"]]
df.loc["row_label"]
df.iloc[0]
df.loc[:, ["A", "B"]]
df.iloc[:, [0, 1]]

# Фильтрация
df[df["A"] > 1]
df[(df["A"] > 1) & (df["B"] < 10)]

# Новый столбец
df["C"] = df["A"] + df["B"]
df["flag"] = df["A"] > 0

# Несколько новых столбцов
df = df.assign(C=lambda x: x["A"] + x["B"])

# Объединение
pd.concat([df1, df2])
pd.concat([df1, df2], axis=1)
pd.merge(df1, df2, on="id", how="left")
df1.merge(df2, on="id")
df1.join(df2)
```

Главная идея pandas: **данные связаны индексами и именами столбцов**.
